<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/presion_atmosferica_IDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Presión atmosférica — estaciones IDEAM cercanas al ROI de la laguna

**Fuente:** `Presión_Atmosférica_20260818_SOLO_BOLIVAR.csv` (IDEAM, departamento de Bolívar)

## Resultados calculados para esta base

Con el archivo entregado:

| Métrica | Resultado |
|---|---:|
| Filas crudas | 1,172,175 |
| Filas limpias después de desduplicar | 1,032,624 |
| Duplicados removidos | 139,551 |
| Valores fuera del rango QC o no numéricos | 5,305 |
| Estaciones únicas | 12 |
| Sitios físicos únicos después de colapsar coordenadas | 11 |

Las dos estaciones/sitios más cercanos al ROI son:

| Estación | Código(s) incluidos | Distancia al centro | Distancia aproximada al borde |
|---|---|---:|---:|
| AEROPUERTO RAFAEL NUNEZ | 0014015080, 0014015020 | 9.47 km | 8.96 km |
| UNIVERSIDAD UNAD CARTAGENA - AUT | 1206500136 | 14.13 km | 13.67 km |

> Nota: `0014015080` y `0014015020` se colapsan como un mismo sitio físico por coordenadas
> prácticamente idénticas en el aeropuerto Rafael Núñez.

## 1. Configuración

In [ ]:
import csv
import io
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cambia esta ruta si mueves el CSV.
RUTA_CSV = Path("/content/Presión_Atmosférica_20260818_SOLO_BOLIVAR.csv")

FORMATO_FECHA = "%Y %b %d %I:%M:%S %p"

# Conservar como texto para no perder ceros a la izquierda.
DTYPES_TEXTO = {"CodigoEstacion": "string", "CodigoSensor": "string"}

COLS_TEXTO = [
    "CodigoEstacion", "CodigoSensor", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "DescripcionSensor", "UnidadMedida",
]

# Rango plausible amplio para presión atmosférica en hPa.
# Captura centinelas y errores claros sin recortar variabilidad meteorológica normal.
RANGO_PRESION_VALIDO = (850.0, 1100.0)

# ROI de la laguna. Coordenadas en orden longitud, latitud.
ROI_COORDS = [
    [-75.476052, 10.517524],
    [-75.476117, 10.518747],
    [-75.473158, 10.519223],
    [-75.470516, 10.525108],
    [-75.469572, 10.524876],
    [-75.471686, 10.518916],
    [-75.468394, 10.517219],
    [-75.468952, 10.516459],
]

FACTORES = [1, 2, 3, 4]
N_ESTACIONES = 2
TOL_SITIO = 3

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

## 2. Carga del CSV crudo

El archivo nuevo sí puede leerse como CSV normal separado por comas. La función mantiene una ruta
de respaldo para archivos IDEAM que lleguen con cada línea completa encerrada como un único campo.
`df_raw` se conserva inmutable: las transformaciones se hacen sobre copias.

In [ ]:
def cargar_crudo(ruta: Path) -> pd.DataFrame:
    # Carga el CSV IDEAM y conserva códigos con ceros a la izquierda.
    # Se agrega `on_bad_lines='warn'` para manejar líneas mal formadas sin detener el proceso,
    # y `engine='python'` para una mayor robustez en el análisis de archivos CSV.
    df = pd.read_csv(ruta, dtype=DTYPES_TEXTO, encoding="utf-8-sig", on_bad_lines='warn', engine='python')

    # Respaldo para archivos doblemente entrecomillados.
    if df.shape[1] == 1:
        with open(ruta, encoding="utf-8-sig", newline="") as f:
            lineas = [fila[0] for fila in csv.reader(f) if fila]
        df = pd.read_csv(io.StringIO("\n".join(lineas)), dtype=DTYPES_TEXTO)

    return df

df_raw = cargar_crudo(RUTA_CSV)
print(f"{len(df_raw):,} filas x {df_raw.shape[1]} columnas")
df_raw.head(3)

In [ ]:
for col in ["UnidadMedida", "DescripcionSensor", "CodigoSensor", "Departamento"]:
    vals = sorted(df_raw[col].dropna().astype(str).unique().tolist())
    print(f"{col:20s} ({len(vals):>3}): {vals[:8]}{' ...' if len(vals) > 8 else ''}")

print()
print(df_raw.dtypes)

## 3. Limpieza

Operaciones:

1. Normalizar texto: quitar espacios de extremos y colapsar espacios internos.
2. Parsear fecha con formato explícito.
3. Convertir `ValorObservado`: los valores vienen como texto con coma de miles (`1,009`).
4. Marcar como `NaN` presiones fuera del rango plausible.
5. Desduplicar por `(CodigoEstacion, FechaObservacion)`, no por sensor, para evitar duplicados
   del mismo instante reportados bajo etiquetas equivalentes.

In [ ]:
def limpiar(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    for c in COLS_TEXTO:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

    df["FechaObservacion"] = pd.to_datetime(
        df["FechaObservacion"], format=FORMATO_FECHA, errors="coerce"
    )

    df["ValorObservado"] = pd.to_numeric(
        df["ValorObservado"].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )
    df["Latitud"] = pd.to_numeric(df["Latitud"], errors="coerce")
    df["Longitud"] = pd.to_numeric(df["Longitud"], errors="coerce")

    lo, hi = RANGO_PRESION_VALIDO
    df.loc[~df["ValorObservado"].between(lo, hi), "ValorObservado"] = np.nan

    df = (
        df.sort_values(["CodigoEstacion", "FechaObservacion", "CodigoSensor"], kind="stable")
        .drop_duplicates(subset=["CodigoEstacion", "FechaObservacion"], keep="first")
        .reset_index(drop=True)
    )
    return df


df = limpiar(df_raw)
print(f"crudo  : {len(df_raw):,}")
print(f"limpio : {len(df):,}  ({len(df_raw) - len(df):,} duplicados eliminados)")
print(f"NaN QC : {df['ValorObservado'].isna().sum():,} valores fuera de {RANGO_PRESION_VALIDO}")
df.head()

## 4. Catálogo de estaciones

In [ ]:
def moda_determinista(s: pd.Series):
    m = s.dropna().mode()
    return m.sort_values().iat[0] if len(m) else pd.NA


def construir_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    conteo = (
        df.groupby(["CodigoEstacion", "NombreEstacion"], as_index=False)
        .size()
        .rename(columns={"size": "_n"})
    )
    nombre_canonico = (
        conteo.sort_values(
            ["CodigoEstacion", "_n", "NombreEstacion"],
            ascending=[True, False, True],
            kind="stable",
        )
        .drop_duplicates("CodigoEstacion", keep="first")
        .drop(columns="_n")
    )

    agregados = (
        df.groupby("CodigoEstacion")
        .agg(
            Municipio=("Municipio", moda_determinista),
            ZonaHidrografica=("ZonaHidrografica", moda_determinista),
            Latitud=("Latitud", "median"),
            Longitud=("Longitud", "median"),
            n_variantes_nombre=("NombreEstacion", "nunique"),
            n_sensores=("CodigoSensor", "nunique"),
            n_obs=("ValorObservado", "size"),
        )
        .reset_index()
    )
    return nombre_canonico.merge(agregados, on="CodigoEstacion", how="left")


catalogo = construir_catalogo(df)
print(f"{len(catalogo)} estaciones únicas")
catalogo.sort_values("n_obs", ascending=False)

## 5. Geometría, cajas y distancias

In [ ]:
_lons = [c[0] for c in ROI_COORDS]
_lats = [c[1] for c in ROI_COORDS]
CX, CY = (min(_lons) + max(_lons)) / 2, (min(_lats) + max(_lats)) / 2
HW, HH = (max(_lons) - min(_lons)) / 2, (max(_lats) - min(_lats)) / 2
BBOXES = {f: (CX - HW * f, CY - HH * f, CX + HW * f, CY + HH * f) for f in FACTORES}

print(f"Centro del ROI: lat={CY:.6f}, lon={CX:.6f}")
for f in FACTORES:
    x0, y0, x1, y1 = BBOXES[f]
    ancho = (x1 - x0) * 111.32 * math.cos(math.radians(CY))
    alto = (y1 - y0) * 111.32
    print(f"{f}x -> {ancho:5.2f} km x {alto:5.2f} km; area aprox. {ancho * alto:5.2f} km2")

In [ ]:
def dist_haversine_km(lat, lon, lat0: float, lon0: float, R: float = 6371.0088) -> np.ndarray:
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    dphi = np.radians(lat0 - lat)
    dlam = np.radians(lon0 - lon)
    a = (
        np.sin(dphi / 2) ** 2
        + np.cos(np.radians(lat)) * math.cos(math.radians(lat0)) * np.sin(dlam / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))


def clasificar_por_caja(df_est: pd.DataFrame) -> pd.Series:
    nivel = pd.Series(pd.NA, index=df_est.index, dtype="Int32")
    for f in sorted(FACTORES, reverse=True):
        x0, y0, x1, y1 = BBOXES[f]
        dentro = df_est["Longitud"].between(x0, x1) & df_est["Latitud"].between(y0, y1)
        nivel = nivel.mask(dentro, f)
    return nivel


def _xy_local(lon, lat):
    km_lat = 111.32
    km_lon = 111.32 * math.cos(math.radians(CY))
    return np.array([(lon - CX) * km_lon, (lat - CY) * km_lat], dtype=float)


def _dist_punto_segmento(p, a, b) -> float:
    ab = b - a
    den = float(np.dot(ab, ab))
    t = 0.0 if den == 0 else float(np.clip(np.dot(p - a, ab) / den, 0, 1))
    return float(np.linalg.norm(p - (a + t * ab)))


def dist_borde_roi_km(lat: float, lon: float) -> float:
    coords = ROI_COORDS + [ROI_COORDS[0]]
    p = _xy_local(lon, lat)
    return min(
        _dist_punto_segmento(p, _xy_local(*coords[i]), _xy_local(*coords[i + 1]))
        for i in range(len(coords) - 1)
    )


estaciones = catalogo.copy()
estaciones["nivel_caja"] = clasificar_por_caja(estaciones)
estaciones["dist_km"] = dist_haversine_km(
    estaciones["Latitud"], estaciones["Longitud"], CY, CX
).round(2)
estaciones["d_borde_km"] = estaciones.apply(
    lambda r: round(dist_borde_roi_km(r["Latitud"], r["Longitud"]), 2), axis=1
)
estaciones = estaciones.sort_values("dist_km", kind="stable").reset_index(drop=True)

estaciones[[
    "CodigoEstacion", "NombreEstacion", "Municipio", "nivel_caja",
    "dist_km", "d_borde_km", "n_obs"
]].head(12)

## 6. Sitios físicos únicos y selección

In [ ]:
def colapsar_sitios(estaciones: pd.DataFrame, tol: int = TOL_SITIO) -> pd.DataFrame:
    e = estaciones.sort_values("dist_km", kind="stable").copy()
    e["_la"] = e["Latitud"].round(tol)
    e["_lo"] = e["Longitud"].round(tol)

    g = e.groupby(["_la", "_lo"])["CodigoEstacion"]
    e["n_entradas_catalogo"] = g.transform("size")
    e["codigos_del_sitio"] = g.transform(lambda s: [list(s)] * len(s))

    return (
        e.drop_duplicates(["_la", "_lo"], keep="first")
        .drop(columns=["_la", "_lo"])
        .reset_index(drop=True)
    )


sitios = colapsar_sitios(estaciones)
seleccion = sitios.head(N_ESTACIONES).copy()
print(f"{len(estaciones)} entradas de catálogo -> {len(sitios)} sitios físicos")
seleccion[[
    "CodigoEstacion", "NombreEstacion", "Municipio",
    "dist_km", "d_borde_km", "n_obs", "codigos_del_sitio"
]]

## 7. Serie temporal de las estaciones seleccionadas

In [ ]:
mapa_sitio = (
    seleccion[["codigos_del_sitio", "NombreEstacion"]]
    .explode("codigos_del_sitio")
    .rename(columns={"codigos_del_sitio": "CodigoEstacion", "NombreEstacion": "Estacion"})
)
print("Códigos incluidos:", mapa_sitio["CodigoEstacion"].tolist())

serie = (
    df.merge(mapa_sitio, on="CodigoEstacion", how="inner")
    [["Estacion", "CodigoEstacion", "FechaObservacion", "ValorObservado"]]
    .sort_values(["Estacion", "FechaObservacion"], kind="stable")
    .reset_index(drop=True)
)

print(f"{len(serie):,} registros en {serie['Estacion'].nunique()} sitios")
serie.head()

In [ ]:
# Si estás en Colab o Jupyter y no tienes folium:
# !pip install folium

import folium
from folium.plugins import Fullscreen

# Centro del mapa: centro del ROI
mapa = folium.Map(
    location=[CY, CX],
    zoom_start=10,
    tiles="CartoDB positron"
)

Fullscreen().add_to(mapa)

# ROI de la laguna
roi_latlon = [(lat, lon) for lon, lat in ROI_COORDS]

folium.Polygon(
    locations=roi_latlon,
    color="cyan",
    weight=3,
    fill=True,
    fill_opacity=0.15,
    popup="ROI laguna"
).add_to(mapa)

# Cajas anidadas
colores_cajas = {
    1: "red",
    2: "orange",
    3: "yellow",
    4: "green"
}

for f, (x0, y0, x1, y1) in BBOXES.items():
    folium.Rectangle(
        bounds=[(y0, x0), (y1, x1)],
        color=colores_cajas[f],
        weight=2,
        fill=False,
        popup=f"Caja {f}x"
    ).add_to(mapa)

# Marcador del centro del ROI
folium.Marker(
    location=[CY, CX],
    popup="Centro del ROI",
    tooltip="Centro del ROI",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(mapa)

# Estaciones seleccionadas
for _, r in seleccion.iterrows():
    popup = f"""
    <b>{r['NombreEstacion']}</b><br>
    Código principal: {r['CodigoEstacion']}<br>
    Códigos del sitio: {', '.join(r['codigos_del_sitio'])}<br>
    Municipio: {r['Municipio']}<br>
    Latitud: {r['Latitud']:.6f}<br>
    Longitud: {r['Longitud']:.6f}<br>
    Distancia al centro: {r['dist_km']:.2f} km<br>
    Distancia al borde ROI: {r['d_borde_km']:.2f} km<br>
    Observaciones: {r['n_obs']:,}
    """

    folium.Marker(
        location=[r["Latitud"], r["Longitud"]],
        popup=folium.Popup(popup, max_width=350),
        tooltip=r["NombreEstacion"],
        icon=folium.Icon(color="red", icon="cloud")
    ).add_to(mapa)

    # Línea desde el centro del ROI hasta la estación
    folium.PolyLine(
        locations=[[CY, CX], [r["Latitud"], r["Longitud"]]],
        color="purple",
        weight=2,
        opacity=0.8
    ).add_to(mapa)

mapa

## 8. Diagnóstico de cobertura y frecuencia

In [ ]:
def diagnosticar(serie: pd.DataFrame) -> pd.DataFrame:
    s = serie.sort_values(["Estacion", "FechaObservacion"], kind="stable").copy()
    s["_paso_s"] = s.groupby("Estacion")["FechaObservacion"].diff().dt.total_seconds()

    out = (
        s.groupby("Estacion")
        .agg(
            inicio=("FechaObservacion", "min"),
            fin=("FechaObservacion", "max"),
            n_obs=("ValorObservado", "size"),
            n_nulos=("ValorObservado", lambda x: int(x.isna().sum())),
            paso_mediano_s=("_paso_s", "median"),
            paso_minimo_s=("_paso_s", "min"),
            presion_media=("ValorObservado", "mean"),
            presion_min=("ValorObservado", "min"),
            presion_max=("ValorObservado", "max"),
        )
        .reset_index()
    )

    span_s = (out["fin"] - out["inicio"]).dt.total_seconds()
    out["paso_mediano_min"] = (out["paso_mediano_s"] / 60).round(1)
    out["paso_minimo_min"] = (out["paso_minimo_s"] / 60).round(2)
    out["anios"] = (span_s / (365.25 * 86400)).round(2)
    out["pct_completitud"] = (
        out["n_obs"] / (span_s / out["paso_mediano_s"] + 1) * 100
    ).round(1)
    out["presion_media"] = out["presion_media"].round(2)

    return out.drop(columns=["paso_mediano_s", "paso_minimo_s"]).sort_values("inicio")


ventana = diagnosticar(serie)
ventana[[
    "Estacion", "inicio", "fin", "anios", "n_obs", "n_nulos",
    "paso_mediano_min", "paso_minimo_min", "pct_completitud",
    "presion_media", "presion_min", "presion_max",
]]

In [ ]:
cobertura_anual = (
    serie.assign(anio=serie["FechaObservacion"].dt.year)
    .pivot_table(index="anio", columns="Estacion", values="ValorObservado",
                 aggfunc="size", fill_value=0)
)
cobertura_anual

## 9. Serie horaria homogénea

In [ ]:
horaria = (
    serie.dropna(subset=["ValorObservado"])
    .groupby(["Estacion", pd.Grouper(key="FechaObservacion", freq="h")])["ValorObservado"]
    .agg(presion="mean", n_crudos="size")
    .reset_index()
)
horaria["presion"] = horaria["presion"].round(2)

print(f"{len(serie):,} registros crudos -> {len(horaria):,} horas")

resumen_horario = (
    horaria.groupby("Estacion")
    .agg(
        inicio=("FechaObservacion", "min"),
        fin=("FechaObservacion", "max"),
        n_horas=("presion", "size"),
        min_por_hora=("n_crudos", "min"),
        mediana_por_hora=("n_crudos", "median"),
        max_por_hora=("n_crudos", "max"),
    )
    .reset_index()
)
resumen_horario["horas_teoricas"] = (
    (resumen_horario["fin"] - resumen_horario["inicio"]).dt.total_seconds() // 3600 + 1
).astype(int)
resumen_horario["pct_horas_con_dato"] = (
    resumen_horario["n_horas"] / resumen_horario["horas_teoricas"] * 100
).round(1)

resumen_horario[[
    "Estacion", "inicio", "fin", "n_horas", "pct_horas_con_dato",
    "min_por_hora", "mediana_por_hora", "max_por_hora",
]]

## 10. Visualización

In [ ]:
PALETA = ["#D64545", "#2E8B8B", "#5B7FBD", "#C08A2E"]
nombres = horaria["Estacion"].drop_duplicates().tolist()

stats = (
    horaria.groupby("Estacion")["presion"]
    .agg(n="size", media="mean", mediana="median", std="std", minimo="min", maximo="max")
    .round(2)
    .reindex(nombres)
)
display(stats)

fig, axes = plt.subplots(len(nombres), 1, figsize=(15, 3.6 * len(nombres)),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, nombre, color in zip(axes, nombres, PALETA):
    sub = horaria[horaria["Estacion"] == nombre]
    st = stats.loc[nombre]

    ax.plot(sub["FechaObservacion"], sub["presion"], lw=0.45, alpha=0.75, color=color)
    ax.axhspan(st["media"] - st["std"], st["media"] + st["std"],
               color=color, alpha=0.10, zorder=0)
    ax.axhline(st["media"], color="black", lw=1.2, alpha=0.8)
    ax.axhline(st["mediana"], color="black", lw=1.0, ls="--", alpha=0.7)
    ax.axhline(st["minimo"], color="#666", lw=0.8, ls=":", alpha=0.8)
    ax.axhline(st["maximo"], color="#666", lw=0.8, ls=":", alpha=0.8)

    caja = (
        f"n = {int(st['n']):,} h\n"
        f"media = {st['media']:.2f} hPa\n"
        f"mediana = {st['mediana']:.2f} hPa\n"
        f"std = {st['std']:.2f} hPa\n"
        f"rango = {st['minimo']:.2f} - {st['maximo']:.2f} hPa"
    )
    ax.text(0.012, 0.04, caja, transform=ax.transAxes, fontsize=8.5,
            va="bottom", ha="left", family="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                      edgecolor=color, alpha=0.9, linewidth=1.2))

    ax.set_title(nombre, fontsize=10, loc="left", fontweight="bold")
    ax.set_ylabel("hPa")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Fecha")
fig.suptitle("Presión atmosférica horaria — estaciones más cercanas al ROI",
             fontsize=12, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
ciclo = (
    horaria.assign(hora=horaria["FechaObservacion"].dt.hour)
    .groupby(["Estacion", "hora"])["presion"]
    .agg(
        media="mean",
        p10=lambda x: x.quantile(0.10),
        p90=lambda x: x.quantile(0.90),
    )
    .round(2)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
for nombre, color in zip(nombres, PALETA):
    s = ciclo[ciclo["Estacion"] == nombre]
    ax.plot(s["hora"], s["media"], marker="o", ms=4, color=color, label=nombre[:34])
    ax.fill_between(s["hora"], s["p10"], s["p90"], color=color, alpha=0.12)

ax.set_xlabel("Hora local")
ax.set_ylabel("Presión (hPa)")
ax.set_title("Ciclo diario medio de presión atmosférica (banda: p10-p90)")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Limitaciones

**Ausencia de estaciones dentro del ROI.** Ninguna estación cae dentro de las cajas anidadas
definidas a partir del polígono del área de estudio. Las estaciones seleccionadas son las más
cercanas disponibles, no estaciones dentro de la laguna.

**Distancia al borde.** La distancia al centro se calcula con haversine. La distancia al borde
usa una aproximación local plana en kilómetros; para distancias de este orden el error esperado
es pequeño frente a la escala espacial del problema.

**Duplicados y sensores equivalentes.** La base incluye códigos de sensor y grafías de nombre
que pueden representar el mismo sitio o variable. Por eso se desduplica por estación-tiempo y se
colapsan sitios físicos por coordenadas.

**Heterogeneidad temporal.** Rafael Núñez tiene registros subhorarios en parte de la serie. Para
comparaciones entre estaciones conviene usar la serie horaria homogénea, no los registros crudos.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Obtener nombres de estaciones únicos y asignarlos a un índice numérico para el eje Y
estaciones_unicas = horaria["Estacion"].drop_duplicates().tolist()
y_pos = np.arange(len(estaciones_unicas))

# Iterar sobre cada estación para plotear sus puntos de datos
for i, estacion in enumerate(estaciones_unicas):
    df_estacion = horaria[horaria["Estacion"] == estacion]
    ax.plot(df_estacion["FechaObservacion"], [i] * len(df_estacion), 'o', markersize=2, alpha=0.6, label=estacion)

ax.set_yticks(y_pos)
ax.set_yticklabels(estaciones_unicas)
ax.set_xlabel("Fecha de Observación")
ax.set_ylabel("Estación")
ax.set_title("Disponibilidad de datos por estación")
ax.grid(axis='x', alpha=0.75)
plt.tight_layout()
plt.show()